# 🔗 Модуль 2. Урок 3: Все виды JOIN и защита от Cartesian Explosion (Задвоения данных)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_03_joins_deep_dive.ipynb)

---

## 1. Опасность дублирования строк при JOIN

Самая частая ошибка, из-за которой компании увольняют аналитиков: **случайное размножение строк при связи «один-ко-многим»**.

Если у одного клиента 3 заказа, а у каждого заказа по 2 товара:
- `customers` JOIN `orders` -> 3 строки;
- `orders` JOIN `order_items` -> 6 строк.
Если вы после этого сделаете `SUM(orders.delivery_cost)`, стоимость доставки посчитается **дважды для каждого заказа**!


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

# Запрос с правильным подсчетом уникальных заказов и суммы без задвоения
query = """
SELECT 
    c.acquisition_channel,
    COUNT(DISTINCT c.customer_id) AS unique_customers,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.total_amount), 2) AS total_revenue,
    ROUND(SUM(oi.total_amount) / COUNT(DISTINCT o.order_id), 2) AS real_aov
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status = 'Completed'
LEFT JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.acquisition_channel
ORDER BY total_revenue DESC;
"""

print("АККУРАТНЫЙ РАСЧЕТ БЕЗ ДУБЛИРОВАНИЯ МЕТРИК:")
print(pd.read_sql(query, conn))
